# Nano PSM Primary 10M Training

Use this notebook after the debug model has passed validation inspection. It trains the primary Nano PSM config with separate checkpoints from the debug run.

In [ ]:
!pip install -U huggingface_hub datasets safetensors onnx onnxruntime
# Install torch separately if the Colab runtime does not already provide the desired CUDA build.
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
HF_DATASET_REPO = 'chkrishna2001/nano-psm'
HF_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-checkpoints'
DATASET_REVISION = 'main'
# Use a branch or commit that contains the latest Nano PSM training fixes.
CODE_REVISION = 'main'

LOCAL_DATA_DIR = '/content/psm-data'
LOCAL_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-checkpoints'
REPO_DIR = '/content/PSM'


## Upload Gated Dataset

Run this only if you manually uploaded or mounted a locally generated dataset folder into Colab. The folder must contain `train.jsonl`, `validation.jsonl`, `metadata.json`, and report files from the quality gate.

In [ ]:
from huggingface_hub import upload_folder

GATED_LOCAL_FOLDER = '/content/gated-psm-data'  # change if mounted elsewhere

# Uncomment after confirming this folder is from a passed local gate.
# upload_folder(
#     repo_id=HF_DATASET_REPO,
#     repo_type='dataset',
#     folder_path=GATED_LOCAL_FOLDER,
#     path_in_repo='.',
#     commit_message='upload gated nano psm training data'
# )

## Download Dataset And Code

In [ ]:
from huggingface_hub import snapshot_download

dataset_dir = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    revision=DATASET_REVISION,
    local_dir=LOCAL_DATA_DIR,
    force_download=True,
)
print(dataset_dir)

In [ ]:
!git clone https://github.com/chkrishna2001/PSM.git {REPO_DIR} || true
%cd {REPO_DIR}
!git fetch origin
!git checkout {CODE_REVISION}
!git pull --ff-only || true
!git rev-parse --short HEAD


## Preflight

This checks that Colab is using a real ~10M config and the updated training code before the run starts.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

train_path = Path(LOCAL_DATA_DIR) / 'train.jsonl'
validation_path = Path(LOCAL_DATA_DIR) / 'validation.jsonl'
config_path = Path('nano-psm/configs/primary-10m.json')
for required in (train_path, validation_path, config_path):
    assert required.exists(), f'Missing required file: {required}'

sys.path.insert(0, 'nano-psm/src')
from nano_psm.model import NanoPsmConfig
from nano_psm.dataset import load_jsonl

config_doc = json.loads(config_path.read_text(encoding='utf-8'))
model_config = NanoPsmConfig(**config_doc['model'])
parameter_estimate = model_config.estimate_parameters()
print('model', config_doc['name'])
print('parameter_estimate', f'{parameter_estimate:,}')
assert 8_000_000 <= parameter_estimate <= 12_000_000, parameter_estimate

train_examples = load_jsonl(train_path)
validation_examples = load_jsonl(validation_path)
print('train_examples', len(train_examples))
print('validation_examples', len(validation_examples))
assert train_examples, 'No train examples loaded'
assert validation_examples, 'No validation examples loaded'

subprocess.run([sys.executable, '-m', 'compileall', 'nano-psm/src/nano_psm'], check=True)
evaluate_source = Path('nano-psm/src/nano_psm/evaluate.py').read_text(encoding='utf-8')
train_source = Path('nano-psm/src/nano_psm/train.py').read_text(encoding='utf-8')
assert 'score_mae' in evaluate_source
assert 'compute_action_weights' in train_source
print('preflight ok')


## Resume Checkpoints

In [ ]:
from huggingface_hub import snapshot_download

try:
    checkpoint_dir = snapshot_download(
        repo_id=HF_CHECKPOINT_REPO,
        repo_type='model',
        local_dir=LOCAL_CHECKPOINT_DIR,
        resume_download=True,
    )
except Exception as exc:
    print('No checkpoint snapshot yet:', exc)
    checkpoint_dir = LOCAL_CHECKPOINT_DIR

print(checkpoint_dir)

## Train Primary 10M Config

Start with a controlled primary run. Increase `MAX_STEPS` only after metrics, inspection, and checkpoint upload work.

In [ ]:
MAX_STEPS = 2000
EVAL_EVERY = 250
SAVE_EVERY = 250

!python nano-psm/src/nano_psm/train.py \
  --config nano-psm/configs/primary-10m.json \
  --train {LOCAL_DATA_DIR}/train.jsonl \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint-dir {LOCAL_CHECKPOINT_DIR} \
  --resume auto \
  --device auto \
  --max-steps {MAX_STEPS} \
  --eval-every {EVAL_EVERY} \
  --save-every {SAVE_EVERY}


## Review Metrics Tail

Use this immediately after training to confirm validation rows include `score_mae`.


In [ ]:
import json
from pathlib import Path

metrics_path = Path(LOCAL_CHECKPOINT_DIR) / 'metrics.jsonl'
if not metrics_path.exists():
    print('No metrics file yet:', metrics_path)
else:
    rows = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for row in rows[-10:]:
        print(json.dumps(row, indent=2))


## Evaluate Best Checkpoint


In [ ]:
!python nano-psm/src/nano_psm/evaluate.py \
  --config nano-psm/configs/primary-10m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto


## Inspect Validation Mistakes

Run this after evaluation. If this returns an empty list, rerun with `--show-correct` to sample successful predictions.

In [ ]:
!python nano-psm/src/nano_psm/inspect_predictions.py \
  --config nano-psm/configs/primary-10m.json \
  --validation {LOCAL_DATA_DIR}/validation.jsonl \
  --checkpoint {LOCAL_CHECKPOINT_DIR}/checkpoint-best.pt \
  --device auto \
  --limit 30


## Upload Checkpoints

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(
    repo_id=HF_CHECKPOINT_REPO,
    repo_type='model',
    private=True,
    exist_ok=True,
)

upload_folder(
    repo_id=HF_CHECKPOINT_REPO,
    repo_type='model',
    folder_path=LOCAL_CHECKPOINT_DIR,
    path_in_repo='.',
    commit_message='sync nano psm checkpoint'
)